In [ ]:
import rasterio
import numpy as np
import os
import matplotlib.pyplot as plt
import typing

from torch.utils.data import Dataset, DataLoader
from skimage.measure import shannon_entropy

In [ ]:
DATA_PATH = os.getcwd() + "/DATA/"
SPLIT = "train/"
PRE_EVENT_PATH = f"{DATA_PATH}{SPLIT}pre-event"
POST_EVENT_PATH = f"{DATA_PATH}{SPLIT}post-event"
TARGET_PATH = f"{DATA_PATH}{SPLIT}target"



## Hyperparameter
VALID_THR = 0.25
CHANGE_THR = 0.001
ENTROPY_THR = 0.20

In [ ]:
class ChangeDetctionDataset(Dataset):
    def __init__(self,
                 pre_img_path: str,
                 post_img_path: str,
                 target_img_path: str,
                 patch_size: int,
                 stride: int):
        
        self.pre_img_path = pre_img_path
        self.post_img_path = post_img_path
        self.target_img_path = target_img_path
        self.patch_size = patch_size
        self.stride = stride
        self.height = 1024
        self.width = 1024


        self.pre_files = sorted(os.listdir(self.pre_img_path))
        self.post_files = sorted(os.listdir(self.post_img_path))
        self.target_files = sorted(os.listdir(self.target_img_path))

        self.coords = self._generate_patch_coordinates()
        self.transform = None
    
    def _generate_patch_coordinates(self):
        coords = []
        for idx,_ in enumerate(range(len(self.pre_files))):
            for y in range(0, self.height - self.patch_size + 1, self.stride):
                for x in range(0, self.width - self.patch_size + 1, self.stride):
                    coords.append((idx, x, y))
        return coords

    def _read_patch(self, folder_path, filename, x, y):
        file_path = os.path.join(folder_path, filename)
        with rasterio.open(file_path) as src:
            window = rasterio.windows.Window(x, y, self.patch_size, self.patch_size)
            data = src.read(window=window)
            if src.nodata is not None:
                data = np.where(data == src.nodata, 0, data)
            data = np.nan_to_num(data, nan=0.0)

        data = np.transpose(data, (1, 2, 0)).astype(np.float32)
        return data
    
    def patch_filtering(self, pre_file_path,post_file_path, target_file_path, x, y, idx):
        informative_patches = []
        trivial_patches = []
        hard_negative_patches = []
        discard_patches=[]
        ### Validity Threshold on EO Images
        with rasterio.open(pre_file_path) as src:
            window = rasterio.windows.Window(x, y, self.patch_size, self.patch_size)
            data = src.read(window=window, masked=True)
            valid_mask = ~np.ma.getmaskarray(data)
            valid_mask = np.all(valid_mask, axis=0)
            valid_ratio = valid_mask.mean()

            if valid_ratio < VALID_THR:
                discard_patches.append(idx)

        ### Change Ratio Threshold on EO image and Target
        with rasterio.open(target_file_path) as label:
            window = rasterio.windows.Window(x, y, self.patch_size, self.patch_size)
            target = label.read(window=window)
            binary_mask = (target > 0).astype(np.uint8)
            changed_pixel = binary_mask.sum()
            change_ratio = (changed_pixel/max(1, valid_mask.sum()))

            if change_ratio <= CHANGE_THR:
                trivial_patches.append(idx)
            
            elif change_ratio > CHANGE_THR:
                informative_patches.append(idx)

        
        ### Entropy Threshold on SAR imagery only
        with rasterio.open(post_file_path) as src2:
            window = rasterio.windows.Window(x, y, self.patch_size, self.patch_size)
            post = src2.read(window=window)
            entropy = shannon_entropy(post)

            if entropy > ENTROPY_THR:
                hard_negative_patches.append(idx)
        
        return informative_patches, trivial_patches, hard_negative_patches, discard_patches
        
    def __len__(self):
        return len(self.coords)
    
    def __getitem__(self, idx):
        image_idx, x, y = self.coords[idx]

        pre = self._read_patch(self.pre_img_path, self.pre_files[image_idx], x, y)
        post = self._read_patch(self.post_img_path, self.post_files[image_idx], x, y)
        target = self._read_patch(self.target_img_path, self.target_files[image_idx], x, y)

        if self.transform:
            transformed = self.transform(
                image1=pre,
                image2=post,
                mask=target
            )
            pre = transformed["image1"]
            post = transformed["image2"]
            target = transformed["mask"]
        
        return pre, post, target

